In [ ]:
import geopandas as gpd
from pathlib import Path

# =========================
# 1. RUTAS
# =========================

carpeta = Path(r"D:\monsp\Documents\Escuela\Maestria\Eventos\CLEI 2026\02.- Recursos")

areas_verdes_path = carpeta / "areas_verdes.shp"
suelo_conservacion_path = carpeta / "suelo_de_conservacion.shp"
clasificacion_path = carpeta / "Clasificacion_CDMX_recortada.shp"
edificios_path = carpeta / "edificios_cdmx.shp"

# =========================
# 2. CARGAR ARCHIVOS
# =========================

areas_verdes = gpd.read_file(areas_verdes_path)
suelo_conservacion = gpd.read_file(suelo_conservacion_path)
clasificacion = gpd.read_file(clasificacion_path)
edificios = gpd.read_file(edificios_path)

# =========================
# 3. DEFINIR CRS
# =========================

crs_area = "EPSG:32614"

# Si algún archivo no tiene CRS, se asigna EPSG:32614
if areas_verdes.crs is None:
    areas_verdes = areas_verdes.set_crs(crs_area)

if suelo_conservacion.crs is None:
    suelo_conservacion = suelo_conservacion.set_crs(crs_area)

if clasificacion.crs is None:
    clasificacion = clasificacion.set_crs(crs_area)

if edificios.crs is None:
    edificios = edificios.set_crs(crs_area)

# Asegurar que todos estén en EPSG:32614
areas_verdes = areas_verdes.to_crs(crs_area)
suelo_conservacion = suelo_conservacion.to_crs(crs_area)
clasificacion = clasificacion.to_crs(crs_area)
edificios = edificios.to_crs(crs_area)

print("CRS areas_verdes:", areas_verdes.crs)
print("CRS suelo_de_conservacion:", suelo_conservacion.crs)
print("CRS clasificacion:", clasificacion.crs)
print("CRS edificios:", edificios.crs)

# =========================
# 4. CORREGIR GEOMETRÍAS
# =========================

areas_verdes["geometry"] = areas_verdes.geometry.buffer(0)
suelo_conservacion["geometry"] = suelo_conservacion.geometry.buffer(0)
clasificacion["geometry"] = clasificacion.geometry.buffer(0)
edificios["geometry"] = edificios.geometry.buffer(0)

# =========================
# 5. CREAR BASE: ÁREAS VERDES + SUELO DE CONSERVACIÓN
# =========================

base = gpd.GeoDataFrame(
    geometry=list(areas_verdes.geometry) + list(suelo_conservacion.geometry),
    crs=crs_area
)

# Unir todo en una sola geometría
base_union = base.dissolve()

# Área inicial = 100%
area_inicial_ha = base_union.geometry.area.sum() / 10_000

print("\n==============================")
print("ÁREA BASE INICIAL")
print("==============================")
print(f"Área inicial total: {area_inicial_ha:,.2f} ha")
print("Porcentaje inicial: 100.00%")

# =========================
# 6. FILTRAR CLASE 0
# =========================

clasificacion["clase"] = clasificacion["clase"].astype(int)
clasificacion_0 = clasificacion[clasificacion["clase"] == 0].copy()

# Disolver clase 0 para evitar restas innecesarias por polígonos separados
clasificacion_0_union = clasificacion_0.dissolve()

# =========================
# 7. RESTAR CLASE 0
# =========================

restante_1 = gpd.overlay(
    base_union,
    clasificacion_0_union,
    how="difference"
)

area_restante_1_ha = restante_1.geometry.area.sum() / 10_000
area_restada_clase0_ha = area_inicial_ha - area_restante_1_ha
porcentaje_restado_clase0 = (area_restada_clase0_ha / area_inicial_ha) * 100
porcentaje_restante_1 = (area_restante_1_ha / area_inicial_ha) * 100

print("\n==============================")
print("RESTA DE CLASIFICACIÓN CLASE 0")
print("==============================")
print(f"Área restada por clase 0: {area_restada_clase0_ha:,.2f} ha")
print(f"Porcentaje restado por clase 0: {porcentaje_restado_clase0:.2f}%")
print(f"Área restante después de clase 0: {area_restante_1_ha:,.2f} ha")
print(f"Porcentaje restante después de clase 0: {porcentaje_restante_1:.2f}%")

# =========================
# 8. RESTAR EDIFICIOS
# =========================

edificios_union = edificios.dissolve()

restante_final = gpd.overlay(
    restante_1,
    edificios_union,
    how="difference"
)

area_final_ha = restante_final.geometry.area.sum() / 10_000
area_restada_edificios_ha = area_restante_1_ha - area_final_ha
porcentaje_restado_edificios = (area_restada_edificios_ha / area_inicial_ha) * 100

area_total_restada_ha = area_inicial_ha - area_final_ha
porcentaje_total_restado = (area_total_restada_ha / area_inicial_ha) * 100
porcentaje_final_restante = (area_final_ha / area_inicial_ha) * 100

print("\n==============================")
print("RESTA DE EDIFICIOS")
print("==============================")
print(f"Área restada por edificios: {area_restada_edificios_ha:,.2f} ha")
print(f"Porcentaje restado por edificios respecto al área inicial: {porcentaje_restado_edificios:.2f}%")

print("\n==============================")
print("RESULTADO FINAL")
print("==============================")
print(f"Área inicial total: {area_inicial_ha:,.2f} ha")
print(f"Área total restada: {area_total_restada_ha:,.2f} ha")
print(f"Porcentaje total restado: {porcentaje_total_restado:.2f}%")
print(f"Área final restante: {area_final_ha:,.2f} ha")
print(f"Porcentaje final restante: {porcentaje_final_restante:.2f}%")

# =========================
# 9. EXPORTAR RESULTADOS
# =========================

salida_base = carpeta / "base_areas_verdes_suelo_conservacion.shp"
salida_resta_clase0 = carpeta / "resultado_sin_clase0.shp"
salida_final = carpeta / "resultado_final_sin_clase0_ni_edificios.shp"

base_union.to_file(salida_base, encoding="utf-8")
restante_1.to_file(salida_resta_clase0, encoding="utf-8")
restante_final.to_file(salida_final, encoding="utf-8")

print("\n==============================")
print("ARCHIVOS EXPORTADOS")
print("==============================")
print(salida_base)
print(salida_resta_clase0)
print(salida_final)

c:\Users\monsp\AppData\Local\Programs\Python\Python314\Lib\site-packages\pyogrio\raw.py:200: RuntimeWarning: D:\monsp\Documents\Escuela\Maestria\Eventos\CLEI 2026\02.- Recursos\areas_verdes.shp contains polygon(s) with rings with invalid winding order. Autocorrecting them, but that shapefile should be corrected using ogr2ogr for example.
  return ogr_read(


CRS areas_verdes: EPSG:32614
CRS suelo_de_conservacion: EPSG:32614
CRS clasificacion: EPSG:32614
CRS edificios: EPSG:32614

ÁREA BASE INICIAL
Área inicial total: 94,758.87 ha
Porcentaje inicial: 100.00%

RESTA DE CLASIFICACIÓN CLASE 0
Área restada por clase 0: 10,664.64 ha
Porcentaje restado por clase 0: 11.25%
Área restante después de clase 0: 84,094.23 ha
Porcentaje restante después de clase 0: 88.75%

RESTA DE EDIFICIOS
Área restada por edificios: 0.00 ha
Porcentaje restado por edificios respecto al área inicial: 0.00%

RESULTADO FINAL
Área inicial total: 94,758.87 ha
Área total restada: 10,664.64 ha
Porcentaje total restado: 11.25%
Área final restante: 84,094.23 ha
Porcentaje final restante: 88.75%

ARCHIVOS EXPORTADOS
D:\monsp\Documents\Escuela\Maestria\Eventos\CLEI 2026\02.- Recursos\base_areas_verdes_suelo_conservacion.shp
D:\monsp\Documents\Escuela\Maestria\Eventos\CLEI 2026\02.- Recursos\resultado_sin_clase0.shp
D:\monsp\Documents\Escuela\Maestria\Eventos\CLEI 2026\02.- Recurs